# 08 · Modelo final

Ridge sobre o desvio em relação ao último dia, treinado com todos os meses. A faixa de incerteza vem dos erros do Ridge no teste da etapa 07.

In [ ]:
import sys, json, pickle
from pathlib import Path

if Path("/kaggle/input").exists():
    RAIZ = next(Path("/kaggle/input").rglob("src/soja/__init__.py")).parents[2]
else:
    RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from soja import config, experimento

ETAPA = "08_modelo_final"
PASTA = config.pasta_resultados(ETAPA)
plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": .3})

tabela = pd.read_csv(config.ler_resultado("05_tabela", "tabela_mensal.csv"), index_col="mes")
tabela.index = pd.PeriodIndex(tabela.index, freq="M")
COLUNAS = json.loads(config.ler_resultado("06_features", "config.json").read_text())["colunas_finais"]
tabela["desvio"] = tabela["alvo"] - tabela["fim_vs_media"]
print(len(COLUNAS), "features:", COLUNAS)

## 1. Treino com todos os meses

In [ ]:
treino = tabela[COLUNAS + ["desvio"]].dropna()
modelo = experimento.ridge().fit(treino[COLUNAS], treino["desvio"])
ridge = modelo[-1]
print(f"{len(treino)} meses ({treino.index[0]} a {treino.index[-1]}), alpha = {ridge.alpha_:.2f}")

# coeficientes na escala padronizada: quanto o desvio muda com 1 desvio-padrão da feature
coef = pd.Series(ridge.coef_, index=COLUNAS).sort_values()
coef.plot.barh(figsize=(7, 4), color=np.where(coef > 0, "tab:green", "tab:red"), title="Peso de cada feature (padronizado)")
plt.axvline(0, c="k", lw=.8); plt.tight_layout(); plt.savefig(PASTA / "coeficientes.png", dpi=120); plt.show()

## 2. Incerteza a partir dos erros no teste

In [ ]:
p07 = pd.read_csv(config.ler_resultado("07_modelos", "previsoes.csv"))
p07 = p07[(p07["modelo"] == "ridge") & (p07["periodo"] == "teste")]
erros = p07["alvo"] - p07["previsao"]
NIVEIS = {"80%": (0.10, 0.90), "95%": (0.025, 0.975)}
faixas = {n: [float(erros.quantile(a)), float(erros.quantile(b))] for n, (a, b) in NIVEIS.items()}
print(f"{len(erros)} meses de teste; faixas do erro em log:", {n: np.round(f, 4) for n, f in faixas.items()})

plt.hist(erros, bins=30, color="lightgray", edgecolor="gray")
for n, (a, b) in faixas.items():
    plt.axvline(a, ls="--", c="tab:red" if n == "95%" else "tab:orange", label=n); plt.axvline(b, ls="--", c="tab:red" if n == "95%" else "tab:orange")
plt.legend(); plt.title("Erros do Ridge no teste (alvo − previsão)"); plt.show()

## 3. Previsão do próximo mês

In [ ]:
ultimo = tabela[COLUNAS].dropna().index[-1]
desvio = float(modelo.predict(tabela.loc[[ultimo], COLUNAS])[0])
variacao = float(tabela.at[ultimo, "fim_vs_media"]) + desvio
preco_atual = float(tabela.at[ultimo, "preco_usd_t"])
previsao = {
    "mes_base": str(ultimo), "mes_previsto": str(ultimo + 1), "preco_atual": preco_atual,
    "variacao_log": variacao, "preco_previsto": preco_atual * np.exp(variacao),
    "preco_ultimo_dia": preco_atual * np.exp(float(tabela.at[ultimo, "fim_vs_media"])),
    "faixas": {n: [preco_atual * np.exp(variacao + e) for e in f] for n, f in faixas.items()},
}
print(json.dumps(previsao, indent=2))

hist = tabela.loc[ultimo - 35:ultimo, "preco_usd_t"]
x = (ultimo + 1).to_timestamp()
plt.plot(hist.index.to_timestamp(), hist, "k", label="preço FMI (média mensal)")
for n, cor in [("95%", "mistyrose"), ("80%", "lightsalmon")]:
    a, b = previsao["faixas"][n]
    plt.fill_between([hist.index[-1].to_timestamp(), x], [preco_atual, a], [preco_atual, b], color=cor, label=n)
plt.plot([hist.index[-1].to_timestamp(), x], [preco_atual, previsao["preco_previsto"]], "o-", c="tab:red", label="previsão")
plt.ylabel("US$/t"); plt.legend(loc="upper left"); plt.title(f"Previsão para {previsao['mes_previsto']}")
plt.savefig(PASTA / "previsao.png", dpi=120); plt.show()

## 4. Salvando

In [ ]:
with open(PASTA / "modelo.pkl", "wb") as f:
    pickle.dump(modelo, f)
(PASTA / "previsao.json").write_text(json.dumps(previsao, indent=2))
pd.DataFrame({"feature": COLUNAS, "coeficiente": ridge.coef_, "media": modelo[0].mean_, "desvio_padrao": modelo[0].scale_}
             ).to_csv(PASTA / "coeficientes.csv", index=False)
config.salvar_config(PASTA, colunas=COLUNAS, alvo_do_modelo="alvo - fim_vs_media", alpha=float(ridge.alpha_),
                     intercepto=float(ridge.intercept_), meses_treino=[str(treino.index[0]), str(treino.index[-1])],
                     faixas_erro_log=faixas, faixas_de="erros do Ridge no teste da etapa 07")
print(sorted(p.name for p in PASTA.iterdir()))